# Python Concurrency: Threading, Multiprocessing, Asyncio

Everything here is stdlib (plus a benchmark table at the end). The point of each
section is to *see* the GIL's consequences in your own timings.

## 1. CPU-bound benchmark: loop vs. threading vs. multiprocessing

In [ ]:
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

def count_primes(limit):
    count = 0
    for n in range(2, limit):
        for d in range(2, int(n ** 0.5) + 1):
            if n % d == 0:
                break
        else:
            count += 1
    return count

WORK = [30_000] * 8  # 8 identical CPU-bound chunks

def bench(label, fn):
    t0 = time.perf_counter()
    result = fn()
    dt = time.perf_counter() - t0
    print(f"{label:<28} {dt:6.2f}s  (checksum {sum(result)})")
    return dt

results_cpu = {}
results_cpu["sequential"] = bench("sequential loop",
    lambda: [count_primes(w) for w in WORK])
with ThreadPoolExecutor(max_workers=8) as ex:
    results_cpu["threads"] = bench("ThreadPoolExecutor(8)",
        lambda: list(ex.map(count_primes, WORK)))
with ProcessPoolExecutor(max_workers=8) as ex:
    results_cpu["processes"] = bench("ProcessPoolExecutor(8)",
        lambda: list(ex.map(count_primes, WORK)))
# Expect: threads ~= sequential (GIL!), processes ~= sequential / n_cores.

## 2. I/O-bound benchmark: sync vs. threads vs. asyncio

In [ ]:
import asyncio

def fake_io_call(i, delay=0.3):
    time.sleep(delay)          # blocking I/O stand-in (network/DB/disk)
    return i

async def fake_io_call_async(i, delay=0.3):
    await asyncio.sleep(delay) # non-blocking equivalent
    return i

N = 20
results_io = {}

t0 = time.perf_counter()
[fake_io_call(i) for i in range(N)]
results_io["sequential"] = time.perf_counter() - t0

t0 = time.perf_counter()
with ThreadPoolExecutor(max_workers=N) as ex:
    list(ex.map(fake_io_call, range(N)))
results_io["threads"] = time.perf_counter() - t0

t0 = time.perf_counter()
await asyncio.gather(*[fake_io_call_async(i) for i in range(N)])
results_io["asyncio"] = time.perf_counter() - t0

for k, v in results_io.items():
    print(f"{k:<12} {v:5.2f}s")
# Expect: sequential ~= N*delay; threads and asyncio ~= delay (all waits overlap).

## 3. Speedup vs. core count for the process pool

In [ ]:
import os

print("cores available:", os.cpu_count())
baseline = results_cpu["sequential"]
for workers in [1, 2, 4, os.cpu_count()]:
    with ProcessPoolExecutor(max_workers=workers) as ex:
        t0 = time.perf_counter()
        list(ex.map(count_primes, WORK))
        dt = time.perf_counter() - t0
    print(f"workers={workers:<3} {dt:5.2f}s  speedup x{baseline / dt:.1f}")
# Speedup should track physical cores, then flatten (hyperthreads, overhead, Amdahl).

## 4. asyncio with a concurrency limit (semaphore) — the polite-crawler pattern

In [ ]:
async def fetch(url_id, semaphore):
    async with semaphore:                  # at most MAX_CONCURRENT inside
        await asyncio.sleep(0.2)           # swap for httpx.AsyncClient().get(url)
        return f"url_{url_id}: ok"

MAX_CONCURRENT = 5
semaphore = asyncio.Semaphore(MAX_CONCURRENT)

t0 = time.perf_counter()
fetched = await asyncio.gather(*[fetch(i, semaphore) for i in range(20)])
print(f"20 'requests', max {MAX_CONCURRENT} in flight: {time.perf_counter() - t0:.2f}s")
print(fetched[:3])
# 20 tasks / 5 at a time * 0.2s -> ~0.8s. Real use: rate-limiting API calls.

## 5. Mini-project: the benchmark table

In [ ]:
import pandas as pd

table = pd.DataFrame({
    "CPU-bound (s)": {k: round(v, 2) for k, v in results_cpu.items()},
    "I/O-bound (s)": {k: round(results_io.get(k2, float("nan")), 2)
                       for k, k2 in [("sequential", "sequential"),
                                     ("threads", "threads"),
                                     ("processes", "asyncio")]},
})
table.index.name = "approach"
table
# Replace fake_io_call with real HTTP requests (httpx) and count_primes with your own
# CPU-bound workload (image resize, simulation), then update this table.